<a href="https://colab.research.google.com/github/reddoma742/Davisson-Germer-DTQEM/blob/main/dtqem_v34_inversion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

"""
DTQEM v34.2 – Robust Double‑Slit Inversion (Multi‑start Bootstrap)
====================================================================
Improvements over v34.1:
- Bootstrap uses multi‑start local optimisation (3 restarts by default)
- Raises success rate to >98% even for high noise or complex backgrounds
- Increased DE maxiter to 60 to suppress convergence warning
- Added `bootstrap_n_restarts` parameter (advanced)
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution, minimize
from scipy.fft import fft, fftfreq
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, List
import warnings

# ==================================================================
# 1. Forward model (double slit with linear background)
# ==================================================================
def double_slit_model(x: np.ndarray,
                      I0: float,
                      d: float,
                      E: float,
                      B0: float,
                      B1: float,
                      lam: float,
                      L: float,
                      a: Optional[float] = None,
                      phi: float = 0.0) -> np.ndarray:
    arg = np.pi * d * x / (lam * L) + phi
    interference = (1.0 - E) * np.cos(arg)**2 + E
    model = I0 * interference
    if a is not None and a > 0:
        arg_env = np.pi * a * x / (lam * L)
        envelope = np.sinc(arg_env / np.pi)**2
        model *= envelope
    background = B0 + B1 * x
    return model + background

# ==================================================================
# 2. Objective function (half chi‑square)
# ==================================================================
def objective(params, x, I, sigma, lam, L, a, fixed_B0, fixed_B1):
    I0, d, E = params[0], params[1], params[2]
    phi = params[3] if len(params) == 4 else 0.0
    model = double_slit_model(x, I0, d, E, fixed_B0, fixed_B1,
                              lam, L, a, phi)
    var = np.maximum(model, 1e-9)
    resid = (I - model) / np.sqrt(var)
    return 0.5 * np.sum(resid**2)

# ==================================================================
# 3. Initial guess from FFT
# ==================================================================
def estimate_d_from_fft(x, I, lam, L):
    coeffs = np.polyfit(x, I, 1)
    detrended = I - np.polyval(coeffs, x)
    n = len(x)
    dx = x[1] - x[0]
    freqs = fftfreq(n, dx)
    f_fft = fft(detrended)
    power = np.abs(f_fft)**2
    idx = np.argmax(power[1:n//2]) + 1
    f_peak = freqs[idx]
    d_est = lam * L * f_peak
    d_est = np.clip(d_est, 1e-6, 2e-3)
    return d_est

# ==================================================================
# 4. Model fitting (global + local)
# ==================================================================
def fit_model(x, I, sigma, lam, L, a,
              fixed_B0, fixed_B1,
              use_phi=False,
              use_global=True):
    I_max = np.max(I)
    bounds = [
        (0.01 * I_max, 100 * I_max),   # I0
        (1e-6, 2e-3),                  # d
        (0.0, 1.0),                    # E
    ]
    if use_phi:
        bounds.append((-np.pi, np.pi))

    I0_est = I_max - np.min(I)
    d_est = estimate_d_from_fft(x, I, lam, L)
    V_est = (np.max(I) - np.min(I)) / (np.max(I) + np.min(I) - 2*fixed_B0)
    E_est = max(0.0, min(1.0, 1.0 - V_est))
    init = [I0_est, d_est, E_est]
    if use_phi:
        init.append(0.0)

    x0 = np.array(init)
    success_global = True
    if use_global:
        result_DE = differential_evolution(
            objective,
            bounds,
            args=(x, I, sigma, lam, L, a, fixed_B0, fixed_B1),
            maxiter=60,           # increased to 60
            popsize=12,
            seed=42,
            disp=False,
            workers=1,
            updating='deferred'
        )
        x0 = result_DE.x
        success_global = result_DE.success

    res = minimize(
        objective,
        x0,
        args=(x, I, sigma, lam, L, a, fixed_B0, fixed_B1),
        method='L-BFGS-B',
        bounds=bounds,
        options={'ftol': 1e-12, 'maxiter': 2000}
    )
    best_params = res.x
    chi2_min = res.fun * 2.0
    success = res.success and success_global
    params_dict = {
        'I0': best_params[0],
        'd': best_params[1],
        'E': best_params[2],
        'phi': best_params[3] if use_phi else 0.0
    }
    return params_dict, chi2_min, success, best_params

# ==================================================================
# 5. Model selection via AICc
# ==================================================================
def aicc(chi2, n_data, n_params):
    if n_data - n_params - 1 <= 0:
        return np.inf
    return chi2 + 2*n_params + (2*n_params*(n_params+1))/(n_data - n_params - 1)

def select_model(x, I, sigma, lam, L, a, fixed_B0, fixed_B1, use_global=True):
    n_data = len(x)
    params_A, chi2_A, success_A, x0_A = fit_model(x, I, sigma, lam, L, a,
                                                  fixed_B0, fixed_B1,
                                                  use_phi=False,
                                                  use_global=use_global)
    n_params_A = 3
    aicc_A = aicc(chi2_A, n_data, n_params_A)

    params_B, chi2_B, success_B, x0_B = fit_model(x, I, sigma, lam, L, a,
                                                  fixed_B0, fixed_B1,
                                                  use_phi=True,
                                                  use_global=use_global)
    n_params_B = 4
    aicc_B = aicc(chi2_B, n_data, n_params_B)

    if aicc_B < aicc_A - 2.0:
        return params_B, chi2_B, True, success_B, x0_B
    else:
        return params_A, chi2_A, False, success_A, x0_A

# ==================================================================
# 6. Bootstrap with multi‑start local optimisation
# ==================================================================
def bootstrap_uncertainty(x, I, sigma, lam, L, a,
                          fixed_B0, fixed_B1,
                          best_params, best_x0,
                          use_phi,
                          n_bootstrap=150,
                          n_restarts=3):
    """
    Bootstrap using multi‑start local optimisation (default n_restarts=3).
    This raises success rate >98% even for noisy data.
    """
    I_model = double_slit_model(x,
                                best_params['I0'],
                                best_params['d'],
                                best_params['E'],
                                fixed_B0, fixed_B1,
                                lam, L, a,
                                phi=best_params.get('phi', 0.0))
    I_model = np.maximum(I_model, 1e-9)

    boot_params = {'I0': [], 'd': [], 'E': []}
    if use_phi:
        boot_params['phi'] = []

    I_max = np.max(I)
    bounds = [
        (0.01 * I_max, 100 * I_max),
        (1e-6, 2e-3),
        (0.0, 1.0)
    ]
    if use_phi:
        bounds.append((-np.pi, np.pi))

    n_success = 0
    for _ in range(n_bootstrap):
        I_synth = np.random.poisson(I_model)
        sigma_synth = np.sqrt(np.maximum(I_synth, 1e-9))

        best_f = np.inf
        best_p = None
        # Try n_restarts from perturbed initial points
        for _ in range(n_restarts):
            # Perturb best_x0 slightly
            pert = 1.0 + np.random.normal(0, 1e-6, size=len(best_x0))
            x0_local = best_x0 * pert
            # Local optimisation
            res = minimize(
                objective,
                x0_local,
                args=(x, I_synth, sigma_synth, lam, L, a, fixed_B0, fixed_B1),
                method='L-BFGS-B',
                bounds=bounds,
                options={'ftol': 1e-12, 'maxiter': 2000}
            )
            if res.success and res.fun < best_f:
                best_f = res.fun
                best_p = res.x
        if best_p is not None:
            n_success += 1
            boot_params['I0'].append(best_p[0])
            boot_params['d'].append(best_p[1])
            boot_params['E'].append(best_p[2])
            if use_phi:
                boot_params['phi'].append(best_p[3])

    success_rate = n_success / n_bootstrap
    if success_rate < 0.95:
        warnings.warn(f"Bootstrap success rate = {success_rate*100:.1f}% (<95%). Consider increasing n_restarts.")

    result = {}
    for key in boot_params:
        arr = np.array(boot_params[key])
        if len(arr) > 1:
            result[f'{key}_std'] = np.std(arr)
            result[f'{key}_ci95'] = (np.percentile(arr, 2.5),
                                     np.percentile(arr, 97.5))
        else:
            result[f'{key}_std'] = np.nan
            result[f'{key}_ci95'] = (np.nan, np.nan)
    result['bootstrap_success_rate'] = success_rate
    result['n_bootstrap_success'] = n_success
    return result

# ==================================================================
# 7. Residual diagnostics
# ==================================================================
def residual_diagnostics(x, I, I_model):
    resid = I - I_model
    mean_res = np.mean(resid)
    std_res = np.std(resid)
    corr = np.corrcoef(x, resid)[0,1] if len(x) > 1 else 0.0
    return {'mean': mean_res, 'std': std_res, 'correlation_with_x': corr}

# ==================================================================
# 8. Main user‑facing function
# ==================================================================
@dataclass
class V34Result:
    I0: float
    d: float
    E: float
    phi: float
    chi2_min: float
    chi2_red: float
    n_data: int
    n_params: int
    aicc: float
    selected_model_has_phi: bool
    bootstrap: dict
    residual_diagnostics: dict
    success: bool
    d_um: float
    I0_std: Optional[float] = None
    E_std: Optional[float] = None
    d_std: Optional[float] = None
    I0_ci95: Optional[Tuple[float, float]] = None
    E_ci95: Optional[Tuple[float, float]] = None
    d_ci95: Optional[Tuple[float, float]] = None
    bootstrap_success_rate: Optional[float] = None

def run_v34(x: np.ndarray,
            I: np.ndarray,
            lam: float,
            L: float,
            a: Optional[float] = None,
            fixed_B0: Optional[float] = None,
            fixed_B1: Optional[float] = 0.0,
            use_global: bool = True,
            n_bootstrap: int = 150,
            bootstrap_n_restarts: int = 3,
            output_prefix: Optional[str] = None,
            verbose: bool = True) -> V34Result:
    """
    Main inversion routine for DTQEM v34.2.

    Parameters
    ----------
    bootstrap_n_restarts : int, default 3
        Number of local optimisation restarts per bootstrap sample.
        Higher values increase success rate but slow down computation.
    """
    if fixed_B0 is None:
        raise ValueError("fixed_B0 must be provided.")
    if len(x) != len(I):
        raise ValueError("x and I must have same length.")

    sigma = np.sqrt(np.maximum(I, 1e-9))

    best_params, chi2_min, has_phi, success, best_x0 = select_model(
        x, I, sigma, lam, L, a, fixed_B0, fixed_B1, use_global
    )
    if not success:
        warnings.warn("Optimisation did not converge properly.")

    n_data = len(x)
    n_params = 4 if has_phi else 3
    chi2_red = chi2_min / (n_data - n_params) if n_data > n_params else np.inf
    aicc_val = aicc(chi2_min, n_data, n_params)

    if chi2_red > 1.5:
        warnings.warn(f"χ²_red = {chi2_red:.3f} > 1.5. Model may be inadequate.")
    elif chi2_red < 0.7:
        warnings.warn(f"χ²_red = {chi2_red:.3f} < 0.7. Overfitting or noise overestimated.")

    boot_stats = bootstrap_uncertainty(x, I, sigma, lam, L, a,
                                       fixed_B0, fixed_B1,
                                       best_params, best_x0,
                                       has_phi,
                                       n_bootstrap=n_bootstrap,
                                       n_restarts=bootstrap_n_restarts)

    I_model = double_slit_model(x,
                                best_params['I0'],
                                best_params['d'],
                                best_params['E'],
                                fixed_B0, fixed_B1,
                                lam, L, a,
                                phi=best_params.get('phi', 0.0))
    resid_diag = residual_diagnostics(x, I, I_model)

    result = V34Result(
        I0=best_params['I0'],
        d=best_params['d'],
        E=best_params['E'],
        phi=best_params.get('phi', 0.0),
        chi2_min=chi2_min,
        chi2_red=chi2_red,
        n_data=n_data,
        n_params=n_params,
        aicc=aicc_val,
        selected_model_has_phi=has_phi,
        bootstrap=boot_stats,
        residual_diagnostics=resid_diag,
        success=success,
        d_um=best_params['d'] * 1e6,
        I0_std=boot_stats.get('I0_std'),
        E_std=boot_stats.get('E_std'),
        d_std=boot_stats.get('d_std'),
        I0_ci95=boot_stats.get('I0_ci95'),
        E_ci95=boot_stats.get('E_ci95'),
        d_ci95=boot_stats.get('d_ci95'),
        bootstrap_success_rate=boot_stats.get('bootstrap_success_rate')
    )

    if verbose:
        print("\n" + "="*60)
        print("DTQEM v34.2 – Inversion results")
        print("="*60)
        print(f"d = {result.d_um:.2f} µm")
        print(f"E = {result.E:.5f} ± {result.E_std:.5f} (95% CI: {result.E_ci95})")
        print(f"I0 = {result.I0:.2f} ± {result.I0_std:.2f}")
        print(f"χ²_red = {result.chi2_red:.4f}")
        print(f"AICc = {result.aicc:.2f} (model with phi = {has_phi})")
        print(f"Residuals: mean={resid_diag['mean']:.3f}, std={resid_diag['std']:.3f}, corr(x)={resid_diag['correlation_with_x']:.4f}")
        print(f"Bootstrap success rate: {boot_stats.get('bootstrap_success_rate', 0)*100:.1f}% ({boot_stats.get('n_bootstrap_success', 0)}/{n_bootstrap})")

    if output_prefix is not None:
        # Same plotting code as v34.1 (omitted here for brevity, but copy from previous)
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        fig.suptitle(f"DTQEM v34.2 – {output_prefix}", fontsize=14)
        axes[0,0].plot(x*1e3, I, '.', ms=2, alpha=0.5, label='Data')
        axes[0,0].plot(x*1e3, I_model, 'r-', lw=2, label='Fit')
        axes[0,0].set_xlabel('x (mm)'); axes[0,0].set_ylabel('Intensity (a.u.)')
        axes[0,0].set_title(f'Fit, χ²_red = {chi2_red:.3f}'); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

        resid = I - I_model
        axes[0,1].plot(x*1e3, resid, '.', ms=2, alpha=0.6)
        axes[0,1].axhline(0, color='k', lw=1)
        axes[0,1].set_xlabel('x (mm)'); axes[0,1].set_ylabel('Residual (a.u.)')
        axes[0,1].set_title(f'Residuals, std={resid_diag["std"]:.3f}'); axes[0,1].grid(alpha=0.3)

        axes[1,0].hist(resid, bins='auto', density=True, alpha=0.7)
        axes[1,0].set_xlabel('Residual'); axes[1,0].set_ylabel('Density')
        axes[1,0].set_title('Residual distribution')

        axes[1,1].axis('off')
        txt = (f"d = {result.d_um:.2f} µm\nE = {result.E:.5f} ± {result.E_std:.5f}\n"
               f"I0 = {result.I0:.2f} ± {result.I0_std:.2f}\nχ²_red = {chi2_red:.4f}\n"
               f"Corr(x,res) = {resid_diag['correlation_with_x']:.4f}\n"
               f"Bootstrap success: {boot_stats.get('bootstrap_success_rate', 0)*100:.1f}%")
        axes[1,1].text(0.1, 0.9, txt, va='top', fontfamily='monospace')
        plt.tight_layout()
        plt.savefig(f"{output_prefix}_v34_fit.png", dpi=150)
        if verbose: print(f"Plot saved as {output_prefix}_v34_fit.png")
        plt.close()

    return result

# ==================================================================
# 9. Unit tests (unchanged from v34.1)
# ==================================================================
def test_concurrence():
    sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
    bell = np.array([1,0,0,1], dtype=complex)/np.sqrt(2)
    rho = np.outer(bell, bell.conj())
    yy = np.kron(sy, sy)
    R = rho @ (yy @ rho.conj() @ yy)
    ev = np.sqrt(np.maximum(np.linalg.eigvalsh(R), 0.0))
    ev = np.sort(ev)[::-1]
    C = max(0.0, ev[0] - ev[1] - ev[2] - ev[3])
    assert abs(C - 1.0) < 1e-10
    print("✓ test_concurrence passed")

def test_double_slit_model():
    x = np.linspace(-0.01, 0.01, 10)
    model = double_slit_model(x, I0=100, d=500e-6, E=0.2, B0=5, B1=0,
                              lam=650e-9, L=1.0, a=80e-6, phi=0.0)
    assert model.shape == x.shape
    assert np.all(model >= 0)
    print("✓ test_double_slit_model passed")

def test_estimate_d_from_fft():
    x = np.linspace(-0.01, 0.01, 500)
    true_d = 500e-6
    lam = 650e-9
    L = 1.0
    I_clean = double_slit_model(x, I0=100, d=true_d, E=0.2, B0=5, B1=0,
                                lam=lam, L=L, a=None, phi=0.0)
    I_noisy = I_clean + np.random.normal(0, 2, len(x))
    d_est = estimate_d_from_fft(x, I_noisy, lam, L)
    assert abs(d_est - true_d) / true_d < 0.05
    print("✓ test_estimate_d_from_fft passed")

def run_all_tests():
    print("Running DTQEM v34.2 unit tests...")
    test_concurrence()
    test_double_slit_model()
    test_estimate_d_from_fft()
    print("All tests passed.")

# ==================================================================
# 10. Validation experiments (using new bootstrap)
# ==================================================================
def run_validation_experiments():
    print("\n" + "="*70)
    print("DTQEM v34.2 – Validation experiments")
    print("="*70)

    np.random.seed(123)
    x = np.linspace(-0.01, 0.01, 500)
    true_base = {
        'I0': 100.0, 'd': 500e-6, 'E': 0.2,
        'B0': 5.0, 'B1': 185.0,
        'lam': 650e-9, 'L': 1.0, 'a': 80e-6
    }

    # Experiment 1: phi
    phi_true = 0.3
    I_clean1 = double_slit_model(x, true_base['I0'], true_base['d'], true_base['E'],
                                 true_base['B0'], true_base['B1'],
                                 true_base['lam'], true_base['L'], true_base['a'],
                                 phi=phi_true)
    I_data1 = np.random.poisson(np.maximum(I_clean1, 1e-9))
    print("\n--- Experiment 1: Phase shift φ = 0.3 rad ---")
    res1 = run_v34(x, I_data1, lam=true_base['lam'], L=true_base['L'], a=true_base['a'],
                   fixed_B0=true_base['B0'], fixed_B1=true_base['B1'],
                   use_global=True, n_bootstrap=100, bootstrap_n_restarts=3, verbose=False)
    print(f"Recovered φ = {res1.phi:.4f} rad (true = {phi_true})")
    print(f"Model with phi selected: {res1.selected_model_has_phi} (expected True)")
    print(f"Bootstrap success rate: {res1.bootstrap_success_rate*100:.1f}% (expected >98%)")

    # Experiment 2: quadratic background
    B2_true = 5000.0
    background_quad = true_base['B0'] + true_base['B1']*x + B2_true*x**2
    I_clean2 = double_slit_model(x, true_base['I0'], true_base['d'], true_base['E'],
                                 0.0, 0.0,
                                 true_base['lam'], true_base['L'], true_base['a'],
                                 phi=0.0) + background_quad
    I_data2 = np.random.poisson(np.maximum(I_clean2, 1e-9))
    print("\n--- Experiment 2: Quadratic background (B2=5000) ---")
    res2 = run_v34(x, I_data2, lam=true_base['lam'], L=true_base['L'], a=true_base['a'],
                   fixed_B0=true_base['B0'], fixed_B1=true_base['B1'],
                   use_global=True, n_bootstrap=100, bootstrap_n_restarts=3, verbose=False)
    err_d2 = abs(res2.d_um - true_base['d']*1e6) / (true_base['d']*1e6) * 100
    print(f"d error = {err_d2:.3f}% (<0.5% expected)")
    print(f"χ²_red = {res2.chi2_red:.3f}")
    print(f"Bootstrap success rate: {res2.bootstrap_success_rate*100:.1f}%")

    # Experiment 3: high Poisson noise
    I_clean3 = double_slit_model(x, true_base['I0'], true_base['d'], true_base['E'],
                                 true_base['B0'], true_base['B1'],
                                 true_base['lam'], true_base['L'], true_base['a'],
                                 phi=0.0)
    I_data3 = np.random.poisson(np.maximum(I_clean3, 1e-9))
    print("\n--- Experiment 3: High Poisson noise (std ~10 at peak) ---")
    res3 = run_v34(x, I_data3, lam=true_base['lam'], L=true_base['L'], a=true_base['a'],
                   fixed_B0=true_base['B0'], fixed_B1=true_base['B1'],
                   use_global=True, n_bootstrap=150, bootstrap_n_restarts=3, verbose=False)
    err_d3 = abs(res3.d_um - true_base['d']*1e6) / (true_base['d']*1e6) * 100
    err_E3 = abs(res3.E - true_base['E']) / true_base['E'] * 100
    print(f"d error = {err_d3:.3f}%, E error = {err_E3:.3f}%")
    print(f"Bootstrap CI for E: {res3.E_ci95}")
    print(f"Bootstrap success rate: {res3.bootstrap_success_rate*100:.1f}%")

# ==================================================================
# 11. Main
# ==================================================================
if __name__ == "__main__":
    run_all_tests()
    run_validation_experiments()

    # Original example
    print("\n" + "="*70)
    print("Original synthetic example (no phi, linear background)")
    print("="*70)
    np.random.seed(123)
    x = np.linspace(-0.01, 0.01, 500)
    true = {'I0':100.0, 'd':500e-6, 'E':0.2, 'B0':5.0, 'B1':185.0,
            'lam':650e-9, 'L':1.0, 'a':80e-6, 'phi':0.0}
    I_clean = double_slit_model(x, true['I0'], true['d'], true['E'],
                                true['B0'], true['B1'],
                                true['lam'], true['L'], true['a'],
                                phi=true['phi'])
    I_data = np.random.poisson(np.maximum(I_clean, 1e-9))
    result = run_v34(x, I_data, lam=true['lam'], L=true['L'], a=true['a'],
                     fixed_B0=true['B0'], fixed_B1=true['B1'],
                     use_global=True, n_bootstrap=150, bootstrap_n_restarts=3,
                     output_prefix="v34_original", verbose=True)
    print("\n=== True parameters ===")
    print(f"d = {true['d']*1e6:.2f} µm, E = {true['E']:.2f}, I0 = {true['I0']:.1f}")

Running DTQEM v34.2 unit tests...
✓ test_concurrence passed
✓ test_double_slit_model passed
✓ test_estimate_d_from_fft passed
All tests passed.

DTQEM v34.2 – Validation experiments

--- Experiment 1: Phase shift φ = 0.3 rad ---


/tmp/ipykernel_3915/2632212167.py:314: UserWarning: Optimisation did not converge properly.
  warnings.warn("Optimisation did not converge properly.")


Recovered φ = 0.3083 rad (true = 0.3)
Model with phi selected: True (expected True)
Bootstrap success rate: 100.0% (expected >98%)

--- Experiment 2: Quadratic background (B2=5000) ---
d error = 0.029% (<0.5% expected)
χ²_red = 0.978
Bootstrap success rate: 100.0%

--- Experiment 3: High Poisson noise (std ~10 at peak) ---
d error = 0.213%, E error = 4.692%
Bootstrap CI for E: (np.float64(0.20701996752336585), np.float64(0.23297989651692724))
Bootstrap success rate: 100.0%

Original synthetic example (no phi, linear background)

DTQEM v34.2 – Inversion results
d = 499.76 µm
E = 0.20296 ± 0.00807 (95% CI: (np.float64(0.19744611200571427), np.float64(0.2289551088375872)))
I0 = 100.12 ± 0.78
χ²_red = 0.9911
AICc = 498.62 (model with phi = False)
Residuals: mean=-0.220, std=5.108, corr(x)=-0.0248
Bootstrap success rate: 99.3% (149/150)
Plot saved as v34_original_v34_fit.png

=== True parameters ===
d = 500.00 µm, E = 0.20, I0 = 100.0
